In [0]:
import pyspark.sql.functions as F
from datetime import datetime
 
catalog = "workspace"
schema = "stock_analytics"
 
print("=" * 70)
print("DATA QUALITY VALIDATION - FINANCIAL DATA")
print("=" * 70)

In [0]:

# Cell 1: Create Quality Metrics Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.quality_metrics (
    check_id STRING,
    check_name STRING,
    table_name STRING,
    layer STRING,
    total_records INT,
    failed_records INT,
    pass_rate DOUBLE,
    severity STRING,
    check_timestamp TIMESTAMP,
    status STRING
)
USING DELTA
""")
 
print("✅ Quality metrics table created")
 

In [0]:

# Cell 2: Bronze Layer Quality Checks
print("\n" + "=" * 70)
print("BRONZE LAYER QUALITY CHECKS")
print("=" * 70 + "\n")
 
bronze_df = spark.table(f"{catalog}.{schema}.bronze_stock_prices")
total_bronze = bronze_df.count()
 
# Check 1: NULL values
null_check = bronze_df.filter(
    F.col("Close").isNull() | 
    F.col("Volume").isNull() | 
    F.col("Ticker").isNull()
).count()
 
print(f"✅ NULL values: {null_check} issues")
 
# Check 2: Negative prices
negative_prices = bronze_df.filter(
    (F.col("Close") <= 0) | 
    (F.col("High") <= 0) | 
    (F.col("Low") <= 0)
).count()
 
print(f"✅ Negative prices: {negative_prices} issues")
 
# Check 3: Logic: High >= Low
logic_issues = bronze_df.filter(
    (F.col("High") < F.col("Low")) |
    (F.col("High") < F.col("Close")) |
    (F.col("Low") > F.col("Close"))
).count()
 
print(f"✅ Price logic (High<Low): {logic_issues} issues")
 
# Check 4: Volume validation
zero_volume = bronze_df.filter(F.col("Volume") <= 0).count()
 
print(f"✅ Zero volume days: {zero_volume} issues")
 
# Check 5: Duplicates
duplicates = bronze_df.groupBy("Ticker", "Date").count().filter(F.col("count") > 1).count()
 
print(f"✅ Duplicate ticker-date: {duplicates} issues")
 
total_issues = null_check + negative_prices + logic_issues + zero_volume + duplicates
pass_rate = round((total_bronze - total_issues) / total_bronze * 100, 2) if total_bronze > 0 else 0
 
print(f"\n{'✅ PASS' if total_issues == 0 else '⚠️ REVIEW'}: {pass_rate}% pass rate")
 
# COMMAND ----------